# SQLite の FTS5 で日本語の文書をブール検索・フレーズ検索する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/21_search/inverted_index.ipynb)

NLP クックブック 21-01 ／ 検索

手元の日本語の文書を、検索サーバーを立てずに AND / OR / NOT や語順どおりのフレーズで検索したい。Python に組み込まれた SQLite の全文検索 (FTS5) で、文字 3-gram の索引と、分かち書きしてから入れる索引を作り、ヒットの違いを確かめる。

In [1]:
!pip install -q fugashi unidic-lite

In [2]:
import io, re, sqlite3, tarfile, time, unicodedata, urllib.request
import pandas as pd
from fugashi import Tagger

def load_livedoor():
    url = "https://www.rondhuit.com/download/ldcc-20140209.tar.gz"
    with urllib.request.urlopen(url) as r:
        tar = tarfile.open(fileobj=io.BytesIO(r.read()), mode="r:gz")
    rows = []
    for m in tar.getmembers():
        parts = m.name.split("/")
        if m.isfile() and len(parts) == 3 and parts[2] != "LICENSE.txt":
            lines = tar.extractfile(m).read().decode("utf-8").split("\n")
            rows.append({"label": parts[1], "title": lines[2], "text": "\n".join(lines[3:])})
    return pd.DataFrame(rows)

# 所要時間の目安: Colab で 1 分以内 (ダウンロードと読み込み)
# livedoor ニュースコーパス (7,367 記事)。記事は CC BY-ND 2.1 JP (改変禁止) で、索引を作って検索するだけにとどめる
df = load_livedoor()
norm = lambda s: unicodedata.normalize("NFKC", s)   # 全角英数字・半角カナなどをそろえる (文書とクエリの両方に使う)
df["title"], df["body"] = df["title"].map(norm), df.pop("text").map(norm)

tagger = Tagger()
seg = lambda s: [w.surface for w in tagger(s)]
print("SQLite", sqlite3.sqlite_version, "/", len(df), "件")
print(seg("東京都立大学で自然言語処理を研究する"))

SQLite 3.45.1 / 7367 件
['東京', '都立', '大学', 'で', '自然', '言語', '処理', 'を', '研究', 'する']


In [3]:
# 所要時間の目安: Colab で 1〜2 分 (7,367 記事の分かち書きを含む)
con = sqlite3.connect(":memory:")   # ファイル名を渡せば索引をディスクに保存できる
con.execute("CREATE TABLE docs(id INTEGER PRIMARY KEY, label TEXT, title TEXT, body TEXT)")
con.executemany("INSERT INTO docs VALUES (?, ?, ?, ?)", zip(range(len(df)), df["label"], df["title"], df["body"]))

# (1) 文字 3-gram の索引: 分かち書きは不要。本文は docs から読む (content=) ので、索引だけを持つ
t = time.time()
con.execute("CREATE VIRTUAL TABLE tri USING fts5(title, body, content='docs', content_rowid='id', tokenize='trigram')")
con.execute("INSERT INTO tri(tri) VALUES ('rebuild')")   # docs の全行から索引を作る
print(f"trigram の索引: {time.time() - t:.1f} 秒")

# (2) 分かち書きしてから入れる索引: 単語を空白で区切った文字列を入れ、空白で区切る tokenizer (unicode61) に任せる
t = time.time()
con.execute("CREATE VIRTUAL TABLE wak USING fts5(title, body, tokenize='unicode61')")
con.executemany("INSERT INTO wak(rowid, title, body) VALUES (?, ?, ?)",
                ((i, " ".join(seg(a)), " ".join(seg(b))) for i, a, b in zip(range(len(df)), df["title"], df["body"])))
print(f"分かち書きの索引: {time.time() - t:.1f} 秒 (分かち書きを含む)")
for name in ["docs", "tri", "wak"]:   # 表と索引の大きさ (dbstat で集計)
    size = con.execute("SELECT sum(pgsize) FROM dbstat WHERE name = ? OR name GLOB ?", (name, name + "_*")).fetchone()[0]
    print(f"  {name}: {size / 1e6:.0f} MB")

trigram の索引: 8.7 秒
分かち書きの索引: 9.2 秒 (分かち書きを含む)
  docs: 31 MB
  tri: 60 MB
  wak: 51 MB


In [4]:
def to_wakati(query):
    """クエリの "..." の中を、文書と同じように分かち書きする ("携帯電話" → "携帯 電話" のフレーズ)。
    "..." の外は分かち書きしないので、語は必ず "..." で囲む (囲まない 携帯電話 は分かち書きの索引に当たらない)"""
    return re.sub(r'"([^"]*)"', lambda m: '"' + " ".join(seg(m.group(1))) + '"', query)

def search(query, table, k=0):
    q = to_wakati(norm(query)) if table == "wak" else norm(query)
    n = con.execute(f"SELECT count(*) FROM {table} WHERE {table} MATCH ?", (q,)).fetchone()[0]
    if not k:
        return n, []
    # bm25() は小さいほどよく一致する。重みはタイトル 5、本文 1
    top = con.execute(f"""SELECT docs.label, docs.title FROM {table} JOIN docs ON docs.id = {table}.rowid
                          WHERE {table} MATCH ? ORDER BY bm25({table}, 5.0, 1.0) LIMIT ?""", (q, k)).fetchall()
    return n, top

queries = [
    '"iPhone" AND "バッテリー"',
    '"サッカー" AND ("日本代表" OR "なでしこ")',
    '"スマートフォン" NOT "Android"',      # FTS5 の NOT は「A NOT B」の二項演算
    '"携帯電話"',                          # フレーズ: 語がこの順番で隣り合う
    '"携帯" AND "電話"',                    # 語順・位置を問わない
    'NEAR("iPhone" "ケース", 5)',           # 間が 5 トークン以内 (trigram は 1 文字ずつ、分かち書きは 1 語ずつ数える)
    'title : "iPhone"',                     # タイトルの列だけで探す
]
for q in queries:
    (n_tri, top), (n_wak, _) = search(q, "tri", k=2), search(q, "wak")
    print(f"{q}\n   trigram {n_tri} 件 / 分かち書き {n_wak} 件  {to_wakati(norm(q))}")
    for label, title in top:
        print(f"     [{label}] {title[:40]}")

"iPhone" AND "バッテリー"
   trigram 105 件 / 分かち書き 105 件  "iPhone" AND "バッテリー"
     [kaden-channel] iPhone4Sユーザーに朗報! アップルが「iOS5」の修正版配布でバッテリー
     [kaden-channel] iPhoneユーザー予想外! iOSアップデートでもバッテリー問題が解決しない!
"サッカー" AND ("日本代表" OR "なでしこ")
   trigram 236 件 / 分かち書き 236 件  "サッカー" AND ("日本 代表" OR "なでしこ")
     [sports-watch] 【Sports Watch】なでしこ以上に露出した美人解説者、臨時収入は「怖い金
     [sports-watch] 【Sports Watch】美少女テクニシャンに天才ドリブラー、次世代なでしこは
"スマートフォン" NOT "Android"
   trigram 438 件 / 分かち書き 433 件  "スマート フォン" NOT "Android"
     [smax] ワイヤレス・テクノジー・パーク2012:日本無線ブースにてスマートフォンをITS
     [smax] Google、ドラえもんの誕生“100年前”を記念してスマートフォン向けコンテン
"携帯電話"
   trigram 366 件 / 分かち書き 366 件  "携帯 電話"
     [kaden-channel] 携帯電話がお手柄!泥棒の携帯電話が偶然警察に電話をかけ、逮捕へ【話題】
     [smax] 携帯電話会社4社、8月30日から東京メトロ丸ノ内線・日比谷線・千代田線・南北線の
"携帯" AND "電話"
   trigram 0 件 / 分かち書き 390 件  "携帯" AND "電話"
NEAR("iPhone" "ケース", 5)
   trigram 29 件 / 分かち書き 59 件  NEAR("iPhone" "ケース", 5)
     [it-life-hack] iPhone5用ケース続々先行発売中! 存在しないはずのiPhone 5用ケース
     [kaden-channel] 傷が消えるケースは世界初! 日産が開発のiPhon

In [5]:
# 1 語のクエリで、部分文字列としての出現 (str.contains) と比べる
def contains(s):
    return int((df["title"].str.contains(s, regex=False) | df["body"].str.contains(s, regex=False)).sum())

rows = []
for w in ["東京", "京都", "東京都", "スマートフォン", "日本代表", "iPhone", "iphone"]:
    rows.append({"語": w, "分かち書き": " ".join(seg(w)), "部分文字列": contains(w),
                 "trigram": search(f'"{w}"', "tri")[0], "分かち書き索引": search(f'"{w}"', "wak")[0],
                 "分かち書き索引 (末尾を前方一致)": search(f'"{w}" *', "wak")[0]})
print(pd.DataFrame(rows).to_string(index=False))
# 文書の中での区切られ方の例
print(seg("東京都内で"), seg("東京都千代田区"), seg("最新のハイスペックスマートフォン「"))
# 「京都」の部分文字列の一致のうち、「東京都」の一部でないものがある文書の数
no_tokyo = (df["title"] + "\n" + df["body"]).str.replace("東京都", "", regex=False)
print("「東京都」を除いても「京都」を含む文書:", int(no_tokyo.str.contains("京都", regex=False).sum()), "件")

      語    分かち書き  部分文字列  trigram  分かち書き索引  分かち書き索引 (末尾を前方一致)
     東京       東京   1111        0     1111               1111
     京都       京都    361        0       68                 68
    東京都     東京 都    298      298      190                298
スマートフォン スマート フォン   1074     1074     1063               1063
   日本代表    日本 代表    293      293      293                293
 iPhone   iPhone    881      884      884                884
 iphone   iphone      6      884      884                884
['東京', '都内', 'で'] ['東京', '都', '千代田', '区'] ['最新', 'の', 'ハイスペックスマートフォン', '「']
「東京都」を除いても「京都」を含む文書: 68 件


In [6]:
# trigram の索引は 3 文字未満の語を MATCH で探せない。LIKE なら探せるが、3 文字未満のパターンでは索引を使わず全件を調べる
for sql, arg in [("SELECT count(*) FROM tri WHERE tri MATCH ?", '"東京都"'),
                 ("SELECT count(*) FROM tri WHERE body LIKE ?", "%東京都%"),
                 ("SELECT count(*) FROM tri WHERE body LIKE ?", "%東京%"),
                 ("SELECT count(*) FROM wak WHERE wak MATCH ?", '"東京"')]:
    t = time.perf_counter()
    for _ in range(10):
        n = con.execute(sql, (arg,)).fetchone()[0]
    print(f"{(time.perf_counter() - t) / 10 * 1000:7.2f} ms  {n:5d} 件  {sql.split('WHERE')[1].strip()} {arg}")
t = time.perf_counter()
n = int(df["body"].str.contains("東京", regex=False).sum())
print(f"{(time.perf_counter() - t) * 1000:7.2f} ms  {n:5d} 件  pandas の str.contains (本文) 東京")

   0.08 ms    298 件  tri MATCH ? "東京都"
   2.76 ms    298 件  body LIKE ? %東京都%
  53.95 ms   1103 件  body LIKE ? %東京%
   0.11 ms   1111 件  wak MATCH ? "東京"
   8.23 ms   1103 件  pandas の str.contains (本文) 東京


- FTS5 は SQLite に組み込まれた全文検索の機能で、Python の標準ライブラリの sqlite3 から使える。検索サーバーも追加のパッケージも要らず、`sqlite3.connect("ファイル名")` にすれば索引をファイルに保存して持ち運べる。SQLite はパブリックドメイン。trigram の tokenizer は SQLite 3.34.0 以降で使える（Colab の Python の SQLite は上の出力の版）。
- クエリは FTS5 の構文で書き、`AND`、`OR`、`NOT`、かっこ、`"..."` のフレーズ、`NEAR(..., N)`、`列名 : ...`、末尾の `*`（前方一致）が使える。`NOT` は「A NOT B」の二項演算で、単独の否定は書けない。`bm25()` の値は小さい（負に大きい）ほどよく一致し、引数で列ごとの重みを渡せる。分かち書きの索引では、クエリの `"..."` の中も文書と同じ方法で分かち書きしてフレーズにする (`to_wakati`)。`"..."` の外は分かち書きしないので、語は必ず `"..."` で囲む。FTS5 は語の出現位置も索引に持つので、フレーズ `"携帯 電話"`（366 件）と、語順や位置を問わない `"携帯" AND "電話"`（390 件）を区別できる。
- trigram の索引は、3 文字以上の語なら部分文字列の出現と同じ件数がヒットした（東京都 298 件、スマートフォン 1074 件。英字は大文字と小文字を区別しないぶん多くなり、iPhone は部分文字列の 881 件に対して 884 件）。そのかわり、3 文字未満の「東京」「京都」は MATCH で 0 件になり、`"携帯" AND "電話"` のように短い語を含むクエリ全体も 0 件になる。LIKE なら探せるが、3 文字未満のパターンは索引を使わない全件の走査になり、上の出力では `%東京%` (54 ms) が `%東京都%` (2.8 ms) より 1 桁以上遅く、メモリ上の pandas の str.contains (8 ms) よりも遅かった。索引を引く MATCH は 0.1 ms 前後で、全件を調べる方法より桁違いに速い（時間は実行ごとに変わる）。また「京都」を部分文字列で探すと 361 件当たるが、「東京都」を取り除いても「京都」を含む文書は 68 件なので、残りは「東京都」の一部への一致である。
- 分かち書きしてから入れる索引は、2 文字の語も探せて、「京都」は「東京都」に当たらない（68 件で、「東京都」を除いた部分文字列の件数と同じ）。そのかわり、文書の中での区切られ方がクエリと違うと漏れる。「東京都内」は「東京 / 都内」と区切られるので、フレーズ「東京 都」では 190 件にとどまり、末尾を前方一致にした `"東京 都" *` で 298 件に戻る。「スマートフォン」は 11 件漏れ、前方一致でも戻らない。主な原因は、カタカナが長く続く文脈で全体が 1 語（ハイスペックスマートフォン）になることである。
- 2 つの索引は弱点が逆なので、両方を持って結果を合わせる構成が実用的（この章の「単語に区切らず文字 n-gram で検索する（表記ゆれに強い検索）」も参照）。どちらの tokenizer も英字の大文字と小文字を区別しない（既定）ので、「iphone」でも 884 件ヒットする。`NEAR` の距離はトークンの数なので、trigram（1 文字ずつ）と分かち書き（1 語ずつ）では同じ N でも範囲が違う（29 件と 59 件）。
- 複数の人がサーバーとして使う、kuromoji の検索モードや同義語を使う、といった場合は、この章の「Elasticsearch で日本語の全文検索をする (kuromoji)」へ進む。Rust 製の検索ライブラリ Tantivy にも Python 版があり（`pip install tantivy`、MIT ライセンス）、分かち書きした文字列を入れれば日本語にも使えるはずである（ここでは試していない）。
- データは livedoor ニュースコーパス（株式会社ロンウイット配布）。各記事は CC BY-ND 2.1 JP（表示 - 改変禁止）で、ここでは索引を作って検索するだけにしている。fugashi は MIT ライセンス（同梱の MeCab は BSD ライセンス）、辞書の unidic-lite は UniDic（BSD ライセンスなどから選べる）を含む。
- 転置索引を自分で実装して仕組み（出現位置を使ったフレーズ検索など）を確かめる版は、付録「仕組みを理解するための実装」の「転置索引を自作してブール検索・フレーズ検索をする」にある。